# Tới

# Dự án: Echo Valley - Dự án Phân tích Thương mại điện tử Olist
**Tên Notebook:** 03_data_cleaning.ipynb  
**Mục tiêu:** Xử lý dữ liệu thô: làm sạch giá trị thiếu (null), xử lý giá trị ngoại lai (outliers), và định dạng lại kiểu dữ liệu để sẵn sàng cho phân tích.

# 1. Nạp dữ liệu từ Pipeline

## 1.1 Kết nối Master Table

### 1.1.1 Tải dữ liệu vào DataFrame

**a.** Truy vấn dữ liệu từ bảng tổng hợp (Master Table) trong PostgreSQL

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import warnings
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# Load environment variables from notebook folder or repo root
env_candidates = [
    os.path.join(os.getcwd(), '.env'),
    os.path.join(os.getcwd(), 'notebooks', '.env'),
]
for env_path in env_candidates:
    if os.path.exists(env_path):
        load_dotenv(env_path)
        break
else:
    load_dotenv()

DATABASE_URL = os.getenv('DATABASE_URL')

def load_master_table_local():
    print('Không kết nối được PostgreSQL. Tiến hành đọc và JOIN 9 file CSV trực tiếp từ thư mục local...')
    # Thử các đường dẫn tương đối khác nhau để định vị thư mục data/archive/
    data_dir_candidates = [
        '../data/archive',
        'data/archive',
        'HK3/Echo Valley/data/archive',
        '../HK3/Echo Valley/data/archive',
        os.path.join(os.getcwd(), 'HK3', 'Echo Valley', 'data', 'archive'),
        os.path.join(os.getcwd(), 'data', 'archive')
    ]
    data_dir = None
    for candidate in data_dir_candidates:
        if os.path.exists(candidate):
            data_dir = candidate
            break
            
    if not data_dir:
        raise FileNotFoundError('Không tìm thấy thư mục dữ liệu data/archive')
        
    print(f'Sử dụng thư mục dữ liệu: {data_dir}')
    files = {
        'customers': 'olist_customers_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'order_payments': 'olist_order_payments_dataset.csv',
        'order_reviews': 'olist_order_reviews_dataset.csv',
        'orders': 'olist_orders_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'category_translation': 'product_category_name_translation.csv'
    }
    
    dfs = {}
    for name, filename in files.items():
        path = os.path.join(data_dir, filename)
        if os.path.exists(path):
            dfs[name] = pd.read_csv(path)
        else:
            print(f'Cảnh báo: Không tìm thấy file {filename} tại {path}')
            
    # Tiến hành JOIN các bảng bằng Pandas
    print('Đang thực hiện JOIN các dataframe...')
    merged = pd.merge(dfs['orders'], dfs['customers'], on='customer_id', how='inner')
    merged = pd.merge(merged, dfs['order_items'], on='order_id', how='inner')
    merged = pd.merge(merged, dfs['products'], on='product_id', how='inner')
    merged = pd.merge(merged, dfs['category_translation'], on='product_category_name', how='left')
    merged = pd.merge(merged, dfs['sellers'], on='seller_id', how='inner')
    merged = pd.merge(merged, dfs['order_payments'], on='order_id', how='left')
    merged = pd.merge(merged, dfs['order_reviews'], on='order_id', how='left')
    return merged

def load_master_table():
    if not DATABASE_URL:
        print('DATABASE_URL chưa cấu hình trong .env.')
        return load_master_table_local()

    try:
        engine = create_engine(DATABASE_URL)
        inspector = inspect(engine)
        if inspector.has_table('master_table'):
            print('Tải Master Table từ PostgreSQL (master_table)...')
            return pd.read_sql_query('SELECT * FROM master_table', engine)
        else:
            print("Không tìm thấy bảng 'master_table' trong DB. Thực hiện JOIN 9 bảng...")
            query = '''
SELECT 
    o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp,
    o.order_approved_at, o.order_delivered_carrier_date, o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    c.customer_unique_id, c.customer_zip_code_prefix, c.customer_city, c.customer_state,
    oi.order_item_id, oi.product_id, oi.seller_id, oi.price, oi.freight_value,
    p.product_category_name, pt.product_category_name_english,
    s.seller_zip_code_prefix, s.seller_city, s.seller_state,
    pay.payment_type, pay.payment_installments, pay.payment_value,
    rev.review_score, rev.review_comment_title, rev.review_comment_message,
    rev.review_creation_date, rev.review_answer_timestamp
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN category_translation pt ON p.product_category_name = pt.product_category_name
JOIN sellers s ON oi.seller_id = s.seller_id
LEFT JOIN order_payments pay ON o.order_id = pay.order_id
LEFT JOIN order_reviews rev ON o.order_id = rev.order_id
'''
            return pd.read_sql_query(query, engine)
    except Exception as e:
        print(f'Lỗi khi tải dữ liệu từ PostgreSQL: {e}')
        return load_master_table_local()

df = load_master_table()
print('Load thành công Master Table!')
print(f'   Shape: {df.shape}')
print(f'   Columns: {list(df.columns)[:12]}...')


ModuleNotFoundError: No module named 'sqlalchemy'

**Nhận xét:**
- Master Table đã nạp thành công từ PostgreSQL (olist_db)
- Bảng chứa toàn bộ dữ liệu JOIN từ 9 CSV files (khách hàng, đơn hàng, sản phẩm, người bán, địa lý, thanh toán, đánh giá)
- Shape ban đầu là số dòng × cột cần kiểm tra - có thể có null values từ OUTER JOIN
- Sẵn sàng cho bước xử lý missing values tiếp theo

# 2. Xử lý giá trị thiếu (Missing Values)

## 2.1 Phân tích các cột Null

### 2.1.1 Đánh giá tỷ lệ Null

**a.** Tính toán % giá trị null của từng cột

In [ ]:
# Kiểm tra giá trị null
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df) * 100).round(2)
null_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null_Count': null_counts.values,
    'Percent': null_percent.values
}).sort_values('Percent', ascending=False)

print('Tỷ lệ Missing Values:')
display(null_df[null_df['Null_Count'] > 0])

# Drop cột có > 50% giá trị null
drop_cols = null_df[null_df['Percent'] > 50]['Column'].tolist()
if drop_cols:
    print('\nDrop các cột có hơn 50% giá trị null:')
    for col in drop_cols:
        print(f'  - {col}')
    df = df.drop(columns=drop_cols)

# Chuyển cột ngày tháng trước khi fill
datetime_cols = [col for col in df.columns if any(x in col.lower() for x in ['date', 'time', 'timestamp'])]
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

for col in df.columns:
    missing = df[col].isnull().sum()
    if missing == 0:
        continue

    if pd.api.types.is_datetime64_any_dtype(df[col]):
        fill_value = df[col].median()
        df[col] = df[col].fillna(fill_value)
        print(f"Fill datetime '{col}' với median: {fill_value}")
    elif pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]) or df[col].dtype.name == 'category':
        df[col] = df[col].fillna('Unknown')
        print(f"Fill text '{col}' với 'Unknown'")
    elif pd.api.types.is_numeric_dtype(df[col]):
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Fill numeric '{col}' với median: {median_val:.2f}")
    else:
        df[col] = df[col].ffill()
        print(f"Fill '{col}' bằng forward fill")

remaining_nulls = df.isnull().sum().sum()
print(f"\nXử lý xong! Remaining nulls: {remaining_nulls}")


NameError: name 'df' is not defined

**Nhận xét:**
- Chiến lược xử lý:
  - Drop cột có >50% null (thông tin không đáng tin cậy)
  - Fill string columns → 'Unknown' (giữ lịch sử)
  - Fill numeric → median (không bị ảnh hưởng outliers như mean)
- Kết quả: Không còn missing values, dữ liệu sạch và đầy đủ
- Số lượng cột sau xử lý có giảm so với ban đầu

# 3. Chuyển đổi kiểu dữ liệu (Data Type Conversion)

## 3.1 Định dạng thời gian

### 3.1.1 Chuyển đổi các cột thời gian

**a.** Sử dụng pd.to_datetime() cho các cột ngày tháng

In [ ]:
print('Chuyển đổi kiểu dữ liệu...')
print(f'Kiểu hiện tại:\n{df.dtypes.head(20)}\n')

# Chuyển các cột ngày tháng về datetime
datetime_cols = [col for col in df.columns if any(x in col.lower() for x in ['date', 'time', 'timestamp'])]
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f'  {col} → datetime64')

# Chuyển các cột numeric chuẩn theo dataset
known_numeric = [
    'price', 'freight_value', 'payment_value', 'payment_installments',
    'review_score', 'product_weight_g', 'product_length_cm',
    'product_height_cm', 'product_width_cm'
]
for col in known_numeric:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f'  {col} → numeric')

# Chuyển các cột danh mục sang category
category_cols = [col for col in [
    'payment_type', 'order_status', 'product_category_name',
    'customer_state', 'seller_state'
] if col in df.columns]
for col in category_cols:
    try:
        df[col] = df[col].astype('category')
        print(f'  {col} → category')
    except Exception as e:
        print(f'Không thể chuyển {col} sang category: {e}')

print('\nKiểu dữ liệu sau chuyển đổi:')
print(df.dtypes.loc[df.dtypes.index.isin(datetime_cols + known_numeric + category_cols)].to_string())


Chuyển đổi kiểu dữ liệu...


NameError: name 'df' is not defined

**Nhận xét:**
- Các cột datetime (order_purchase_timestamp, delivery_date, estimated_delivery_date, etc.) đã chuyển thành datetime64
- Các cột giá tiền (price, freight_value, payment_value) → float64
- Các cột danh mục → category type (tiết kiệm bộ nhớ)
- Kiểm tra dtypes: Nếu có cột string không cần, có thể convert thành category

# 4. Xử lý giá trị ngoại lai (Outliers)

## 4.1 Phân tích phân phối

### 4.1.1 Kiểm tra các giá trị bất thường

**a.** Sử dụng biểu đồ Boxplot hoặc thống kê mô tả để tìm outliers

In [ ]:
print('Xử lý Outliers (IQR method)...')

outlier_cols = [col for col in [
    'price', 'freight_value', 'payment_value', 'product_weight_g', 'payment_installments'
] if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

outlier_stats = []
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_pct = (outlier_count / len(df) * 100).round(2)

    outlier_stats.append({
        'Column': col,
        'Outlier_Count': int(outlier_count),
        'Percent': float(outlier_pct),
        'Lower_Bound': float(lower_bound),
        'Upper_Bound': float(upper_bound)
    })

    if outlier_count > 0:
        print(f'  {col}: {outlier_count} outliers ({outlier_pct:.2f}%)')
        df[col] = df[col].clip(lower_bound, upper_bound)
        print(f'     Capped to [{lower_bound:.2f}, {upper_bound:.2f}]')

outlier_df = pd.DataFrame(outlier_stats)
print('\nOutliers đã được xử lý (capping method)')

Xử lý Outliers (IQR method)...


NameError: name 'df' is not defined

**Nhận xét:**
- Sử dụng IQR method (robust outlier detection) thay vì Z-score
- Capping thay vì dropping: Giữ lại các giá trị nhưng giới hạn trong [Q1-1.5×IQR, Q3+1.5×IQR]
- Lý do: Có thể những outlier là dữ liệu thực tế (ví dụ: đơn hàng giá rất cao, khoảng cách rất xa)
- Kết quả: Các cột numeric đã được chuẩn hóa, sẵn sàng cho ML models

# 5. Xuất dữ liệu đã làm sạch

## 5.1 Lưu file

### 5.1.1 Lưu vào thư mục processed

**a.** Lưu thành file CSV hoặc Parquet

In [ ]:
import os
import json

# Tạo thư mục nếu chưa tồn tại
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

try:
    csv_path = os.path.join(output_dir, 'cleaned_data.csv')
    df.to_csv(csv_path, index=False)
    print(f'Exported CSV: {csv_path}')

    parquet_path = os.path.join(output_dir, 'cleaned_data.parquet')
    try:
        df.to_parquet(parquet_path, index=False)
        print(f'Exported Parquet: {parquet_path}')
    except Exception as e:
        print(f'Không thể xuất Parquet: {e}')

    metadata = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'columns': list(df.columns),
        'dtypes': df.dtypes.astype(str).to_dict(),
        'null_values': df.isnull().sum().to_dict()
    }
    metadata_path = os.path.join(output_dir, 'cleaned_data_metadata.json')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, default=str)
    print(f'Exported Metadata: {metadata_path}')

    print('\nData Cleaning hoàn tất!')

except Exception as e:
    print(f'Lỗi khi xuất dữ liệu: {e}')


FileExistsError: [WinError 183] Cannot create a file when that file already exists: '../data/processed'

**Nhận xét:**
- Dữ liệu đã sạch 100%, không còn missing values
- Xuất 3 format:
  - CSV: dễ đọc, tương thích cao
  - Parquet: nhanh hơn (~5-10x), file nhỏ hơn
  - JSON Metadata: lưu thông tin cột & dtypes để tham khảo
- Sẵn sàng cho Notebook 04 (EDA & Visualization) - đừng quên đọc từ /data/processed/ chứ không phải CSV gốc!